<a href="https://www.kaggle.com/code/zayyamwaseem/us-accident-severity-smote-xgboost?scriptVersionId=313374025" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🚦 US Accident Severity Prediction: SMOTE & XGBoost Optimization
### *Methodology Improvement Study on 7.7 Million Records*

## 📋 Project Overview
This notebook focuses on predicting road accident severity using the **US Accidents (2016-2023)** dataset. Our goal is to accurately classify the impact of an accident on traffic flow (Severity levels 1-4) while addressing critical gaps in existing research

## 🚩 The Problem (The Gap)
Existing methodologies often achieve high overall accuracy but suffer from **extremely low recall for minority severity classes** (Levels 1 and 4). 
* **The Gap:** Models are typically biased toward the majority class (Level 2), often missing the most critical, high-impact accidents.
* **Improvement:** This notebook prioritizes the **Macro-Averaged

## 🛠️ 1. Data Loading & Efficient Sampling
Given the scale of **7.7 million records** (3.06 GB), we implement memory-efficient loadin. We use stratified sampling to maintain the original class proportions while ensuring the notebook remains responsive for training.

In [1]:
import pandas as pd
import numpy as np
import os

# 1. Find the exact path to avoid FileNotFoundError
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'US_Accidents_March23.csv' in filename:
            path = os.path.join(dirname, filename)
            print(f"File found at: {path}")

# 2. Load the dataset with limited rows for memory efficiency
# Your paper mentions 7.7 million records; we start with 500,000 for the 'improvement' phase
df = pd.read_csv(path, nrows=500000)

print("Dataset successfully loaded!")
print(f"Initial shape: {df.shape}")

File found at: /kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv
Dataset successfully loaded!
Initial shape: (500000, 46)


## 🏗️ 2. Lifecycle Feature Engineering
We move beyond basic spatiotemporal data by extracting **temporal risk indices**:
* **Rush Hour Detection:** Identifying peak volatility periods (7-9 AM and 4-6 PM).
* **Temporal Granularity:** Extracting hour, weekday, and month features to capture seasonal and daily patterns

In [2]:
# Convert to datetime and extract lifecycle features
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
df['Hour'] = df['Start_Time'].dt.hour
df['Weekday'] = df['Start_Time'].dt.weekday
df['Month'] = df['Start_Time'].dt.month

# Create a 'Rush_Hour' feature to capture high-risk periods
df['Rush_Hour'] = df['Hour'].apply(lambda x: 1 if x in [5, 6, 10-13] else 0)

# Drop columns identified as irrelevant in Phase 2 of your paper
cols_to_drop = ['ID', 'Source', 'Description', 'Street', 'City', 'County', 'Zipcode', 
                'Country', 'Timezone', 'Airport_Code', 'Weather_Timestamp', 
                'Wind_Chill(F)', 'Start_Time', 'End_Time']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print("Feature engineering complete. Current columns:", df.columns.tolist())

Feature engineering complete. Current columns: ['Severity', 'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)', 'State', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight', 'Hour', 'Weekday', 'Month', 'Rush_Hour']


## 🧹 3. Advanced Preprocessing & Cleaning
We address the dataset's significant missing data (e.g., **44% null values** in endpoint coordinates). 
* **Threshold-Based Dropping:** Removing columns with >40% missing values to preserve data quality.
* **Imputation:** Applying median and mode imputation to ensure the data is 100% clean for SMOTE.

In [3]:
from sklearn.preprocessing import LabelEncoder

# 1. Drop columns with more than 40% missing values (like End_Lat/End_Lng) 
# This preserves the rest of your 7.7 million records.
limit = len(df) * 0.4
df = df.dropna(thresh=limit, axis=1)

# 2. Advanced Imputation
# Numeric: Fill with Median
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

# Categorical: Fill with the most frequent value (Mode)
for col in df.select_dtypes(include=['object']).columns:
    if not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode())
    else:
        df[col] = df[col].fillna('Unknown')

# 3. Encode Categorical Features safely
le = LabelEncoder()
for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col].astype(str))

# 4. Final cleaning: Drop any remaining rows that still have NaNs
df = df.dropna()

print(f"Preprocessing complete! Samples remaining for training: {len(df)}")

# 5. Define Features and Target
X = df.drop('Severity', axis=1)
y = df['Severity']

Preprocessing complete! Samples remaining for training: 500000


## ⚖️ 4. Class Imbalance Resolution (SMOTE)
To fix the "low recall gap," we apply **Synthetic Minority Over-sampling Technique (SMOTE)**. This balances the training set so the model can effectively learn the characteristics of rare, high-severity events.

## 🚀 5. Optimized XGBoost Training
We train an **XGBoost Classifier** on the balanced data. This ensemble approach captures non-linear relationships between weather, temporal factors, and severity.

In [4]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Split into Training and Testing sets (80/20 split)
# The stratify=y ensures we keep the rare Severity levels in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply SMOTE to fix the 'low recall' issue
print("Applying SMOTE to balance severity levels...")
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print(f"Training set balanced! Each class now has {len(y_resampled)//4} samples.")

Applying SMOTE to balance severity levels...
Training set balanced! Each class now has 249727 samples.


In [5]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# XGBoost expects 0-3 for Severity 1-4
model = XGBClassifier(n_estimators=100, max_depth=6, random_state=42, eval_metric='mlogloss')
model.fit(X_resampled, y_resampled - 1)

# Predict and evaluate
y_pred = model.predict(X_test) + 1
print("Final Results (Methodology Improvement):")
print(classification_report(y_test, y_pred))

Final Results (Methodology Improvement):
              precision    recall  f1-score   support

           1       0.01      0.04      0.01        77
           2       0.87      0.81      0.84     62432
           3       0.72      0.80      0.76     37435
           4       0.02      0.14      0.04        56

    accuracy                           0.81    100000
   macro avg       0.41      0.45      0.41    100000
weighted avg       0.82      0.81      0.81    100000



## 📚 References & Acknowledgments
This methodology is informed by recent research (2024-2025) including work by **Zhou**, **Yamarthi**, **Goubraim**, **Tang**, and **Alotaibi**.

**Dataset Citation:**
> Moosavi, Sobhan, et al. "A Countrywide Traffic Accident Dataset," 2019 [8].